# TESS Exoplanet Pipeline — Step 1: Target Resolution

We import our libraries and resolve the target name to obtain coordinates (RA/Dec) and standardized TIC ID.

In [ ]:
# Enable inline plotting
%matplotlib inline

import os
import re
import csv
import json
import math
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightkurve as lk
import pymc as pm
import arviz as az
import exoplanet as xo
import celerite2.pymc as celerite2_pm
from celerite2.pymc import terms as c2terms
import pytensor.tensor as pt
from astropy.coordinates import SkyCoord
import astropy.units as u
from astroquery.gaia import Gaia
import corner
from transitleastsquares import transitleastsquares

warnings.filterwarnings("ignore")
print("Setup Complete. All standard packages imported without using tess_pipeline.")

In [ ]:
def resolve_target(target):
    raw = str(target).strip()
    match = re.search(r"(?:tic[\s_\-]*)?(\d{6,16})", raw, re.IGNORECASE)
    if not match:
        raise ValueError(f"Cannot parse target '{raw}' as a TIC ID.")
    tic_id = int(match.group(1))
    
    print(f"Querying MAST for TIC {tic_id} coordinates...")
    try:
        search = lk.search_lightcurve(f"TIC {tic_id}", mission="TESS")
        if len(search) == 0:
            print("No MAST observations found; coordinates set to None.")
            return {"tic_id": tic_id, "name": f"TIC {tic_id}", "ra": None, "dec": None}
        t = search.table
        ra = float(t["s_ra"][0]) if "s_ra" in t.colnames else None
        dec = float(t["s_dec"][0]) if "s_dec" in t.colnames else None
        return {"tic_id": tic_id, "name": f"TIC {tic_id}", "ra": ra, "dec": dec}
    except Exception as e:
        print(f"MAST target resolution failed: {e}")
        return {"tic_id": tic_id, "name": f"TIC {tic_id}", "ra": None, "dec": None}

target_info = resolve_target("TIC 261136679")
print("Resolved Target Information:")
for key, value in target_info.items():
    print(f"  {key}: {value}")

## Step 2: Archival Reference Lookup

We query the local NASA Exoplanet Archive TOI export file in the `data/` directory to check if there is a known cataloged period and transit epoch for this target.

In [ ]:
def query_local_archive(tic_id, data_dir="data"):
    csv_files = sorted(Path(data_dir).glob("TOI_*.csv"), reverse=True)
    if not csv_files:
        print("No local TOI catalog CSV found in data/.")
        return {"period": None, "source": "none"}
    
    archive_path = csv_files[0]
    print(f"Querying local TOI catalog: {archive_path.name}")
    
    with archive_path.open("r", encoding="utf-8", newline="") as f:
        filtered = (line for line in f if not line.startswith("#"))
        reader = csv.DictReader(filtered)
        rows = list(reader)
        
    matches = []
    for r in rows:
        for tid_key in ("tid", "ticid", "tic_id"):
            val = r.get(tid_key)
            if val and int(val) == tic_id:
                matches.append(r)
                break
                
    if not matches:
        print(f"No entries found in local TOI catalog for TIC {tic_id}")
        return {"period": None, "source": "none"}
    
    def record_sort_key(row):
        disp = (row.get("tfopwg_disp") or "").strip().upper()
        disp_rank = {"CP": 0, "KP": 1, "PC": 2, "APC": 3, "FA": 4, "FP": 5}.get(disp, 99)
        try:
            pl_pnum = int(float(row.get("pl_pnum", 9999)))
        except:
            pl_pnum = 9999
        try:
            toi = float(row.get("toi", 999999.0))
        except:
            toi = 999999.0
        return (disp_rank, pl_pnum, toi)
        
    matches.sort(key=record_sort_key)
    row = matches[0]
    
    def _safe_float(v):
        if v in (None, ""): return None
        try:
            f = float(v)
            return f if np.isfinite(f) else None
        except:
            return None
            
    period = _safe_float(row.get("pl_orbper"))
    toi = row.get("toi", "")
    planet_name = f"TOI-{toi}" if toi else None
    
    return {
        "period": period,
        "epoch": _safe_float(row.get("pl_tranmid")),
        "planet_name": planet_name,
        "rp_earth": _safe_float(row.get("pl_rade")),
        "t_eq": _safe_float(row.get("pl_eqt")),
        "reference": f"Local TOI catalog export: {archive_path.name}",
        "source": "toi-local",
        "raw_row": row
    }

archive_result = query_local_archive(target_info["tic_id"])
print("Local TOI Catalog Query:")
for key, value in archive_result.items():
    if key != "raw_row":
        print(f"  {key}: {value}")

## Step 3: Data Acquisition

We search and download SPOC PDCSAP light curves from MAST using `lightkurve`. To make analysis reproducible and fast, we download a single sector of data.

In [ ]:
def download_spoc_lightcurves(tic_id, sector=1, cadence=120):
    print(f"Searching MAST for SPOC light curves for TIC {tic_id}...")
    search = lk.search_lightcurve(
        f"TIC {tic_id}",
        author="SPOC",
        exptime=cadence,
        mission="TESS"
    )
    if len(search) == 0:
        raise ValueError(f"No short-cadence SPOC data found for TIC {tic_id}")
    
    table = search.table
    if "sequence_number" in table.colnames:
        sorted_idx = sorted(range(len(search)), key=lambda i: table["sequence_number"][i])
        search = search[sorted_idx]
        
    n_sectors = min(sector, len(search))
    print(f"Downloading {n_sectors} sector(s) of FITS products...")
    
    os.makedirs("data/fits", exist_ok=True)
    collection = search[:n_sectors].download_all(
        flux_column="pdcsap_flux",
        download_dir="data/fits",
        quality_bitmask="hardest"
    )
    
    # Move and flatten downloaded FITS files out of mastDownload structure
    import shutil
    mast_dir = Path("data/fits/mastDownload")
    if mast_dir.exists():
        for fits_file in mast_dir.glob("**/*.fits*"):
            dest = Path("data/fits") / fits_file.name
            shutil.move(str(fits_file), str(dest))
        shutil.rmtree(str(mast_dir))
        
        # Re-read files from flat directory
        flat_files = [str(p) for p in Path("data/fits").glob(f"*-{tic_id}-*.fits*")]
        if not flat_files:
            flat_files = [str(p) for p in Path("data/fits").glob(f"*-{tic_id:016d}-*.fits*")]
        collection = lk.LightCurveCollection([
            lk.read(f, flux_column="pdcsap_flux") for f in flat_files
        ])
        
    return collection

raw_collection = download_spoc_lightcurves(target_info["tic_id"], sector=1)
print(f"Acquired light curve collection containing {len(raw_collection)} sectors.")

## Step 4: Preprocessing & Cleaning

We clean the light curve by removing NaNs and outliers (via sigma-clipping), stitching multiple sectors into a single light curve, and detrending low-frequency stellar variability using a Savitzky-Golay filter.

To avoid deforming or removing the actual transits during outlier rejection and flattening, we generate a transit mask using the known archival period and epoch from Step 2.

In [ ]:
def preprocess_lightcurves(collection, period=None, epoch=None):
    cleaned = []
    for lc in collection:
        lc = lc.remove_nans()
        
        # Construct a transit mask to protect transit points from clipping
        transit_mask = np.zeros(len(lc), dtype=bool)
        if period is not None and epoch is not None:
            t0 = epoch
            if t0 > 2400000 and np.median(lc.time.value) < 100000:
                t0 -= 2457000.0  # Convert BJD to BTJD if needed
            phase = ((lc.time.value - t0) / period) % 1.0
            phase[phase > 0.5] -= 1.0
            # Mask points within 0.15 days (~3.6 hours) of transit center
            transit_mask = np.abs(phase) < (0.15 / period)
            
        # Outlier rejection: only clip positive outliers (like flares)
        # and ignore the transit region to prevent clipping the transit itself.
        if np.any(transit_mask):
            non_transit_indices = np.where(~transit_mask)[0]
            sub_lc = lc[non_transit_indices]
            _, outlier_mask = sub_lc.remove_outliers(sigma_lower=20.0, sigma_upper=5.0, return_mask=True)
            full_outlier_mask = np.zeros(len(lc), dtype=bool)
            full_outlier_mask[non_transit_indices] = outlier_mask
            lc = lc[~full_outlier_mask]
        else:
            lc = lc.remove_outliers(sigma_lower=20.0, sigma_upper=5.0)
            
        cleaned.append(lc)
        
    stitched = lk.LightCurveCollection(cleaned).stitch()
    
    # Re-calculate transit mask on stitched lightcurve for flattening
    transit_mask_stitched = np.zeros(len(stitched), dtype=bool)
    if period is not None and epoch is not None:
        t0 = epoch
        if t0 > 2400000 and np.median(stitched.time.value) < 100000:
            t0 -= 2457000.0
        phase = ((stitched.time.value - t0) / period) % 1.0
        phase[phase > 0.5] -= 1.0
        transit_mask_stitched = np.abs(phase) < (0.15 / period)
        
    # Flatten with transit mask using a larger window_length (801 cadences ~ 27 hours)
    # to make the trend stiffer and reduce edge distortions around systematic dips.
    flat, trend = stitched.flatten(
        window_length=801,
        polyorder=3,
        break_tolerance=5,
        mask=transit_mask_stitched,
        return_trend=True
    )
    
    # Second outlier rejection pass on flattened data (ignoring transit region)
    if np.any(transit_mask_stitched):
        non_transit_indices = np.where(~transit_mask_stitched)[0]
        sub_flat = flat[non_transit_indices]
        _, outlier_mask = sub_flat.remove_outliers(sigma_lower=20.0, sigma_upper=5.0, return_mask=True)
        full_outlier_mask = np.zeros(len(flat), dtype=bool)
        full_outlier_mask[non_transit_indices] = outlier_mask
        flat = flat[~full_outlier_mask]
    else:
        flat = flat.remove_outliers(sigma_lower=20.0, sigma_upper=5.0)
        
    return flat, stitched, trend

lc, stitched_lc, trend_lc = preprocess_lightcurves(
    raw_collection, 
    period=archive_result.get("period"), 
    epoch=archive_result.get("epoch")
)

# Visualize detrending results
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
axes[0].plot(stitched_lc.time.value, stitched_lc.flux.value, ".k", alpha=0.3, label="Stitched PDCSAP")
axes[0].plot(stitched_lc.time.value, trend_lc.flux.value, "-r", linewidth=1.5, label="Savitzky-Golay Trend")
axes[0].set_ylabel("Flux (e-/s)")
axes[0].legend(loc="upper right")
axes[0].set_title("Stitched Raw Light Curve with Detrending Model")

axes[1].plot(lc.time.value, lc.flux.value, ".b", alpha=0.3, label="Flattened Data")
axes[1].set_ylabel("Normalized Flux")
axes[1].set_xlabel("Time (BTJD)")
axes[1].legend(loc="upper right")
axes[1].set_title("Cleaned & Detrended Input for Period Search")
plt.tight_layout()
plt.show()

## Step 5: Archival Transit Verification

Instead of wasting computational resources on a blind Transit Least Squares (TLS) search, we directly retrieve the known exoplanet parameters (period, epoch, depth, and duration) from the archival TOI catalog, phase-fold the light curve, and plot the folded profile to verify the transit shape.

In [ ]:
if archive_result.get("period") is None or archive_result.get("epoch") is None:
    raise ValueError("No archival TOI candidate information found. A blind TLS periodogram search is required.")

print("Retrieving parameters from the archival TOI catalog...")
detection = {
    "period": float(archive_result["period"]),
    "epoch": float(archive_result["epoch"]),
    "duration_hr": float(archive_result["raw_row"]["pl_trandurh"]) if archive_result["raw_row"].get("pl_trandurh") else 2.5,
    "depth": float(archive_result["raw_row"]["pl_trandep"]) / 1e6 if archive_result["raw_row"].get("pl_trandep") else 0.001,
}

# Convert epoch to BTJD if BJD is used
t0 = detection["epoch"]
if t0 > 2400000 and np.median(lc.time.value) < 100000:
    t0 -= 2457000.0
    detection["epoch"] = t0

print("\nArchival Parameters:")
for key, val in detection.items():
    print(f"  {key}: {val}")

# Fold and Bin helper functions
def fold_lightcurve(time, flux, flux_err, period, epoch=None):
    t0 = epoch if epoch is not None else time[0]
    phase = ((time - t0) / period) % 1.0
    phase[phase >= 0.5] -= 1.0
    sort_idx = np.argsort(phase)
    return phase[sort_idx], flux[sort_idx], flux_err[sort_idx]

def bin_lightcurve(phase, flux, flux_err, n_bins=100):
    bin_edges = np.linspace(-0.5, 0.5, n_bins + 1)
    bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
    bin_flux = np.full(n_bins, np.nan)
    bin_err = np.full(n_bins, np.nan)
    for i in range(n_bins):
        mask = (phase >= bin_edges[i]) & (phase < bin_edges[i + 1])
        if mask.sum() == 0:
            continue
        w = 1.0 / flux_err[mask]**2 if np.all(flux_err[mask] > 0) else np.ones(mask.sum())
        bin_flux[i] = np.average(flux[mask], weights=w)
        bin_err[i] = 1.0 / np.sqrt(np.sum(w))
    return bin_centers, bin_flux, bin_err

# Plot phase-folded light curve
err_arr = lc.flux_err.value if lc.flux_err is not None else np.ones_like(lc.flux.value) * 1e-3
ph, fl, er = fold_lightcurve(lc.time.value, lc.flux.value, err_arr, detection["period"], detection["epoch"])
bin_c, bin_f, bin_e = bin_lightcurve(ph, fl, er, n_bins=100)

fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(ph, fl, s=1.0, color="steelblue", alpha=0.3, label="Data Points")
valid_bins = np.isfinite(bin_f)
ax.errorbar(bin_c[valid_bins], bin_f[valid_bins], yerr=bin_e[valid_bins], fmt="o", ms=3, color="navy", elinewidth=0.8, capsize=2, label="Binned")
ax.set_xlabel(f"Phase (P = {detection['period']:.5f} d)")
ax.set_ylabel("Normalized Flux")
ax.set_title("Phase-Folded Light Curve (Archival Parameters)")
ax.legend()
plt.show()

## Step 6: Stellar Characterization

We query Gaia DR3 and the TESS Input Catalog (TIC) at our target coordinates to retrieve physical parameters like effective temperature, radius, and mass. These values constrain our physical planetary radius and stellar mass, and will be used as priors in our MCMC transit fit.

In [ ]:
def get_stellar_properties(tic_id, ra=None, dec=None, local_toi=None):
    # Prioritize querying the TESS Input Catalog (TIC) via MAST
    print(f"Querying TESS Input Catalog (TIC) for TIC {tic_id}...")
    try:
        from astroquery.mast import Catalogs
        result = Catalogs.query_object(f"TIC {tic_id}", catalog="TIC")
        if len(result) > 0:
            row = result[0]
            rad = row.get("rad")
            mass = row.get("mass")
            teff = row.get("Teff")
            if rad is not None and mass is not None:
                rad_err = row.get("e_rad") if "e_rad" in row.colnames else None
                mass_err = row.get("e_mass") if "e_mass" in row.colnames else None
                return {
                    "source": "TIC",
                    "r_star": float(rad),
                    "r_star_err": float(rad_err) if rad_err is not None and not np.isnan(rad_err) else 0.1 * float(rad),
                    "m_star": float(mass),
                    "m_star_err": float(mass_err) if mass_err is not None and not np.isnan(mass_err) else 0.1 * float(mass),
                    "teff": float(teff) if teff is not None else None,
                }
    except Exception as e:
        print(f"TIC query failed: {e}")

    # Fallback 1: Gaia DR3 Cone Search
    if ra is not None and dec is not None:
        print("Falling back to Gaia DR3 cone search...")
        try:
            coord = SkyCoord(ra=ra*u.deg, dec=dec*u.deg, frame="icrs")
            j = Gaia.cone_search_async(coord, radius=2.0*u.arcsec)
            r = j.get_results()
            if len(r) > 0:
                row = r[0]
                rad = row.get("radius_gspphot")
                teff = row.get("teff_gspphot")
                if rad is not None and not np.isnan(rad):
                    rad = float(rad)
                    teff = float(teff) if teff is not None and not np.isnan(teff) else None
                    if teff is not None:
                        mass = (teff / 5777.0)**1.5 * rad**0.1
                    else:
                        mass = rad
                    return {
                        "source": "Gaia DR3",
                        "r_star": rad,
                        "r_star_err": 0.1 * rad,
                        "m_star": mass,
                        "m_star_err": 0.1 * mass,
                        "teff": teff,
                    }
        except Exception as e:
            print(f"Gaia query failed: {e}")

    # Fallback 2: Local TOI Catalog
    if local_toi is not None and local_toi.get("raw_row") is not None:
        print("Falling back to local TOI catalog...")
        row = local_toi["raw_row"]
        rad_str = row.get("st_rad")
        teff_str = row.get("st_teff")
        if rad_str not in (None, ""):
            rad = float(rad_str)
            teff = float(teff_str) if teff_str not in (None, "") else None
            if teff is not None:
                mass = (teff / 5777.0)**1.5 * rad**0.1
            else:
                mass = rad
            return {
                "source": "Local TOI Catalog",
                "r_star": rad,
                "r_star_err": 0.1 * rad,
                "m_star": mass,
                "m_star_err": 0.1 * mass,
                "teff": teff,
            }

    # Raise error instead of using generic solar parameters
    raise ValueError("Stellar properties could not be resolved from TIC, Gaia, or local catalog. Scientific parameters must be supplied manually.")

stellar_properties = get_stellar_properties(
    target_info["tic_id"], 
    ra=target_info.get("ra"), 
    dec=target_info.get("dec"), 
    local_toi=archive_result
)

# Calculate stellar density in g/cm^3 and propagate errors
m_star = stellar_properties.get("m_star")
r_star = stellar_properties.get("r_star")
m_star_err = stellar_properties.get("m_star_err", 0.1) or 0.1
r_star_err = stellar_properties.get("r_star_err", 0.1) or 0.1

if m_star is not None and r_star is not None:
    rho_star_val = 1.408 * m_star / (r_star**3)
    rho_star_err = rho_star_val * np.sqrt((m_star_err / m_star)**2 + (3.0 * r_star_err / r_star)**2)
    stellar_properties["rho_star"] = float(rho_star_val)
    stellar_properties["rho_star_err"] = float(rho_star_err)
else:
    stellar_properties["rho_star"] = None
    stellar_properties["rho_star_err"] = None

print("Retrieved Stellar Properties:")
for key, value in stellar_properties.items():
    print(f"  {key}: {value}")

## Step 7: Bayesian Transit Modeling

We compile the transit model using `exoplanet` Keplerian orbit structures and `LimbDarkLightCurve`, with a Gaussian Process (GP) noise kernel (`celerite2` SHO) to model stellar noise and systematics. 

To speed up the MCMC sampling, we crop the light curve to a +/- 0.5-day window around each transit center, reducing the number of data points. We sort the cropped data and remove duplicate timestamps to satisfy celerite2's strict sorting requirement, preventing PyTensor assertion crashes.

In [21]:
# Unpack arrays from preprocessed light curve
time_raw = np.asarray(lc.time.value, dtype=float)
flux_raw = np.asarray(lc.flux.value, dtype=float)
flux_err_raw = np.asarray(lc.flux_err.value, dtype=float) if lc.flux_err is not None else np.ones_like(flux_raw) * 1e-3

# Crop data around transit centers to speed up PyMC modeling
t0_prior = detection["epoch"]
period_prior = detection["period"]

t_min, t_max = np.min(time_raw), np.max(time_raw)
n_transit_min = int(np.floor((t_min - t0_prior) / period_prior))
n_transit_max = int(np.ceil((t_max - t0_prior) / period_prior))
transit_times = t0_prior + np.arange(n_transit_min, n_transit_max + 1) * period_prior

# Mask points within 0.5 days of any transit center
in_transit_window = np.zeros_like(time_raw, dtype=bool)
for t_mid in transit_times:
    in_transit_window |= np.abs(time_raw - t_mid) < 0.5

time_arr = time_raw[in_transit_window]
flux_arr = flux_raw[in_transit_window]
flux_err_arr = flux_err_raw[in_transit_window]

# Sort and remove duplicates to guarantee celerite2's strictly sorted, unique time requirement
sort_idx = np.argsort(time_arr)
time_arr = time_arr[sort_idx]
flux_arr = flux_arr[sort_idx]
flux_err_arr = flux_err_arr[sort_idx]

unique_mask = np.concatenate(([True], np.diff(time_arr) > 0))
time_arr = time_arr[unique_mask]
flux_arr = flux_arr[unique_mask]
flux_err_arr = flux_err_arr[unique_mask]

print(f"Reduced data from {len(time_raw)} to {len(time_arr)} points by cropping to +/- 0.5 days around transits.")

# Estimate initial planet/star radius ratio from depth
depth_estimate = float(1.0 - np.percentile(flux_arr, 1))
rp_init = math.sqrt(max(1e-5, depth_estimate))

# Construct explicit initial values dictionary to bypass random jitter violations
init_dict = {
    "period": period_prior,
    "t0": t0_prior,
    "log_rp": np.log(rp_init),
    "b": 0.1,
    "q1": 0.3,
    "q2": 0.3,
    "mean_flux": np.array([0.0]),
    "log_jitter": -6.0,
    "log_sigma_gp": -3.0,
    "log_rho_gp": np.log(5.0),
}
rho_star_val = stellar_properties.get("rho_star")
rho_star_err = stellar_properties.get("rho_star_err")
if rho_star_val is not None and rho_star_val > 0:
    init_dict["rho_star"] = rho_star_val

print(f"Building PyMC model for TIC {target_info['tic_id']}...")
with pm.Model() as model:
    # 1. Stellar Priors
    if rho_star_val is not None and rho_star_val > 0:
        rho_err = rho_star_err if rho_star_err is not None else rho_star_val * 0.1
        rho_star = pm.TruncatedNormal("rho_star", mu=rho_star_val, sigma=rho_err, lower=0.0, initval=rho_star_val)
    else:
        rho_star = pm.LogNormal("rho_star", mu=np.log(1.4), sigma=1.0, initval=1.4)
        
    # 2. Orbital Priors
    period = pm.Normal("period", mu=period_prior, sigma=period_prior * 0.005, initval=period_prior)
    t0 = pm.Normal("t0", mu=t0_prior, sigma=0.05, initval=t0_prior)
    
    # 3. Shape Priors
    log_rp = pm.Uniform("log_rp", lower=np.log(0.001), upper=np.log(0.5), initval=np.log(rp_init))
    rp_r_star = pm.Deterministic("rp_r_star", pm.math.exp(log_rp))
    b = pm.Uniform("b", lower=0.0, upper=1.0 + rp_r_star, initval=0.1)
    
    # Kipping (2013) Limb Darkening Parameterization
    q1 = pm.Uniform("q1", lower=0.0, upper=1.0, initval=0.3)
    q2 = pm.Uniform("q2", lower=0.0, upper=1.0, initval=0.3)
    sqrt_q1 = pm.math.sqrt(q1)
    u1 = pm.Deterministic("u1", 2.0 * sqrt_q1 * q2)
    u2 = pm.Deterministic("u2", sqrt_q1 * (1.0 - 2.0 * q2))
    
    # 4. Systematics and Jitter
    mean_flux = pm.Normal("mean_flux", mu=0.0, sigma=0.01, shape=1, initval=[0.0])
    log_jitter = pm.Normal("log_jitter", mu=-6.0, sigma=2.0, initval=-6.0)
    jitter = pm.Deterministic("jitter", pm.math.exp(log_jitter))
    
    # 5. Keplerian Orbit Setup
    orbit = xo.orbits.KeplerianOrbit(
        period=period,
        t0=t0,
        b=b,
        rho_star=rho_star,
        ror=rp_r_star
    )
    
    # 6. Light Curve Model
    star_lc = xo.LimbDarkLightCurve(u1, u2)
    light_curves = star_lc.get_light_curve(orbit=orbit, r=rp_r_star, t=time_arr)
    transit_model = pm.Deterministic("transit_model", pm.math.sum(light_curves, axis=-1) + mean_flux[0])
    
    # 7. GP Noise Model (celerite2 SHO Kernel)
    log_sigma_gp = pm.Normal("log_sigma_gp", mu=-3.0, sigma=2.0, initval=-3.0)
    log_rho_gp = pm.Normal("log_rho_gp", mu=np.log(5.0), sigma=2.0, initval=np.log(5.0))
    sigma_gp = pm.Deterministic("sigma_gp", pm.math.exp(log_sigma_gp))
    rho_gp = pm.Deterministic("rho_gp", pm.math.exp(log_rho_gp))
    
    term = c2terms.SHOTerm(sigma=sigma_gp, rho=rho_gp, Q=1.0 / np.sqrt(2.0))
    gp = celerite2_pm.GaussianProcess(term, mean=transit_model)
    gp.compute(time_arr, diag=flux_err_arr**2 + pm.math.exp(2.0 * log_jitter), quiet=True)
    gp.marginal("obs", observed=flux_arr)
    
    gp_pred = pm.Deterministic("gp_pred", gp.predict(flux_arr - transit_model))
    flux_model = pm.Deterministic("flux_model", transit_model + gp_pred)
    
    # Dense phase grid model for plotting uncertainty bands
    phase_grid = np.linspace(-0.3, 0.3, 200)
    lc_pred_curves = star_lc.get_light_curve(orbit=orbit, r=rp_r_star, t=t0 + phase_grid)
    pm.Deterministic("lc_pred", pm.math.sum(lc_pred_curves, axis=-1) + 1.0)
    
    # Run NUTS Sampling
    # Using 500 tune steps and 500 draws on 2 chains.
    # We bypass MAP optimization and initialize directly from our valid physical parameters
    # with init="adapt_diag" to prevent random jitter boundary violations.
    print("Sampling parameter posteriors via NUTS MCMC...")
    trace = pm.sample(
        draws=500,
        tune=500,
        chains=2,
        init="adapt_diag",
        initvals=init_dict,
        target_accept=0.9,
        return_inferencedata=True,
        progressbar=True
    )
    print("Sampling complete!")

ValueError: Not enough samples to build a trace.

## Step 8: Convergence Diagnostics & Parameter Derivation

We evaluate the convergence of our MCMC chains (using R-hat, effective sample size, and number of divergences) and derive the physical parameters of the planet candidate (such as planetary radius in Earth radii, semi-major axis, and equilibrium temperature) by propagating the joint posterior uncertainties along with the stellar parameter uncertainties.

In [22]:
summary = az.summary(trace, round_to=6)

# 1. Convergence stats
rhat_col = "r_hat" if "r_hat" in summary.columns else None
rhat_max = max(summary[rhat_col]) if rhat_col else float("nan")

ess_col = "ess_bulk" if "ess_bulk" in summary.columns else None
ess_min = min(summary[ess_col]) if ess_col else float("nan")

divergences = 0
try:
    divergences = int(trace.sample_stats["diverging"].values.sum())
except:
    pass
    
print("Convergence Diagnostics:")
print(f"  R-hat Max: {rhat_max:.4f}")
print(f"  ESS Min  : {ess_min:.1f}")
print(f"  Diverges : {divergences}")

# 2. Extract posterior samples
posterior = trace.posterior
period_samples = posterior["period"].values.flatten()
b_samples = posterior["b"].values.flatten()
rp_r_star_samples = posterior["rp_r_star"].values.flatten()
rho_star_samples = posterior["rho_star"].values.flatten()
t0_samples = posterior["t0"].values.flatten()

# Calculate semi-major axis a/R* for each sample using Kepler's Third Law in CGS
# Constant factor: (G * (86400)^2) / (3 * pi) = 52.864
a_r_star_samples = (52.864 * rho_star_samples * period_samples**2)**(1.0/3.0)

# Calculate transit duration t14 (in hours) for each sample
ratio_term = np.sqrt(np.maximum(1e-6, (1.0 + rp_r_star_samples)**2 - b_samples**2)) / np.sqrt(np.maximum(1e-6, a_r_star_samples**2 - b_samples**2))
t14_samples_days = (period_samples / np.pi) * np.arcsin(np.clip(ratio_term, 0.0, 1.0))
t14_samples_hr = t14_samples_days * 24.0

# Propagate stellar parameters uncertainties
r_star_val = stellar_properties["r_star"]
r_star_err = stellar_properties.get("r_star_err", 0.1 * r_star_val) or 0.1 * r_star_val
r_star_samples = np.random.normal(r_star_val, r_star_err, size=len(rp_r_star_samples))

# Planetary radius in Earth radii (1 R_sun = 109.2 R_earth)
rp_earth_samples = rp_r_star_samples * r_star_samples * 109.2

# Semi-major axis in AU (1 AU = 215.03 R_sun)
a_au_samples = (a_r_star_samples * r_star_samples) / 215.03

# Equilibrium temperature (assuming zero albedo)
teff_val = stellar_properties.get("teff")
if teff_val is not None:
    teff_err = stellar_properties.get("teff_err", 100.0) or 100.0
    teff_samples = np.random.normal(teff_val, teff_err, size=len(a_r_star_samples))
    teq_samples = teff_samples * (0.25 / a_r_star_samples**2)**0.25
else:
    teq_samples = None

# Summarize results with medians and 1-sigma uncertainties
def print_derived_parameter(name, samples):
    med = np.percentile(samples, 50)
    plus = np.percentile(samples, 84) - med
    minus = med - np.percentile(samples, 16)
    print(f"  {name:<25}: {med:.5f} (+{plus:.5f} / -{minus:.5f})")

print("\nDerived Planetary Parameters:")
print_derived_parameter("Period (days)", period_samples)
print_derived_parameter("Epoch (BTJD)", t0_samples)
print_derived_parameter("rp_r_star", rp_r_star_samples)
print_derived_parameter("Impact Parameter (b)", b_samples)
print_derived_parameter("Stellar Density (g/cm^3)", rho_star_samples)
print_derived_parameter("a/R_star", a_r_star_samples)
print_derived_parameter("Transit Duration (hr)", t14_samples_hr)
print_derived_parameter("Planet Radius (R_earth)", rp_earth_samples)
print_derived_parameter("Semi-major axis (AU)", a_au_samples)
if teq_samples is not None:
    print_derived_parameter("Equilibrium Temp (K)", teq_samples)

# Store derived samples for plotting
derived_results = {
    "a_r_star": a_r_star_samples,
    "t14_hr": t14_samples_hr,
    "rp_earth": rp_earth_samples,
    "a_au": a_au_samples,
    "t_eq": teq_samples
}

NameError: name 'trace' is not defined